In [ ]:
#@title Copyright 2023 Google LLC. Double-click here for license information.
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# AII501NAA – Activity 3: Linear Regression

**Student:** Joshua-Grace Lima

This is the *Linear Regression* programming exercise from Google's Machine Learning Crash Course
(`linear_regression_taxi.ipynb`), completed, with one extra section added at the end:

* **Parts 1–4** – the original exercise. The only `TODO` in it (Experiment 2, the hyperparameter
  experiments) is completed as three separate runs: 2a, 2b and 2c.
* **Part 5** – added for the Activity 3 questions: a train/test split, MAE and MSE on the test set,
  a predicted vs. actual plot, one changed setting, and a feature-importance comparison.

The written answers to the eight questions are in `answers.md` in the same folder.

# Colabs

Machine Learning Crash Course uses Colaboratories (Colabs) for all programming exercises. Colab is Google's implementation of [Jupyter Notebook](https://jupyter.org/). For more information about Colabs and how to use them, go to [Welcome to Colaboratory](https://research.google.com/colaboratory).

# Linear Regression
In this Colab you will use a real dataset to train a model to predict the fare of a taxi ride in Chicago, Illinois.

## Learning Objectives
After completing this Colab, you'll be able to:

  * Read a .csv file into a [pandas](https://developers.google.com/machine-learning/glossary/#pandas) DataFrame.
  * Explore a [dataset](https://developers.google.com/machine-learning/glossary/#data_set) with Python visualization libraries.
  * Experiment with different [features](https://developers.google.com/machine-learning/glossary/#feature) to build a linear regression model.
  * Tune the model's [hyperparameters](https://developers.google.com/machine-learning/glossary/#hyperparameter).
  * Compare training runs using [root mean squared error](https://developers.google.com/machine-learning/glossary/#root-mean-squared-error-rmse) and [loss curves](https://developers.google.com/machine-learning/glossary/#loss-curve).

## Dataset Description
The [dataset for this exercise](https://storage.mtls.cloud.google.com/mlcc-nextgen-internal/chicago_taxi_train.csv) is derived from the [City of Chicago Taxi Trips dataset](https://data.cityofchicago.org/Transportation/Taxi-Trips/wrvz-psew). The data for this exercise is a subset of the Taxi Trips data, and focuses on a two-day period in May of 2022.

# Part 1 - Setup Exercise


---

## Load required modules

This exercise depends on several Python libraries to help with data manipulation, machine learning tasks, and data visualization.

**Instructions**
1. Run the **Install required libraries** code cell (below).
1. Run the **Load dependencies** code cell (below).

In [ ]:
#@title Install required libraries

!pip install google-ml-edu==0.1.3 \
  keras~=3.8.0 \
  matplotlib~=3.10.0 \
  numpy~=2.0.0 \
  pandas~=2.2.0 \
  tensorflow~=2.18.0

print('\n\nAll requirements successfully installed.')

In [ ]:
#@title Code - Load dependencies

# data
import numpy as np
import pandas as pd

# machine learning
import keras
import ml_edu.experiment
import ml_edu.results

# data visualization
import plotly.express as px

## Load the dataset


The following code cell loads the dataset and creates a pandas DataFrame.

You can think of a DataFrame like a spreadsheet with rows and columns. The rows represent individual data examples, and the columns represent the attributes associated with each example.

In [ ]:
# @title
# Uses a local copy of the dataset if one is sitting next to this notebook,
# otherwise downloads it from Google (this is what happens in Colab).
import os

DATASET_URL = "https://download.mlcc.google.com/mledu-datasets/chicago_taxi_train.csv"
LOCAL_COPY = "chicago_taxi_train.csv"

dataset_source = LOCAL_COPY if os.path.exists(LOCAL_COPY) else DATASET_URL
chicago_taxi_dataset = pd.read_csv(dataset_source)
print("Loaded dataset from:", dataset_source)

## Update the dataframe

The following code cell updates the DataFrame to use only specific columns from the dataset.

Notice that that output shows just a sample of the dataset, but there should be enough information for you to identify the features associated with the dataset, and have a look at the actual data for a few examples.

In [ ]:
#@title Code - Read dataset

# Updates dataframe to use specific columns.
training_df = chicago_taxi_dataset.loc[:, ('TRIP_MILES', 'TRIP_SECONDS', 'FARE', 'COMPANY', 'PAYMENT_TYPE', 'TIP_RATE')]

print('Read dataset completed successfully.')
print('Total number of rows: {0}\n\n'.format(len(training_df.index)))
training_df.head(200)

# Part 2 - Dataset Exploration


---

## View dataset statistics

A large part of most machine learning projects is getting to know your data. In this step, you will use the ``DataFrame.describe`` method to view descriptive statistics about the dataset and answer some important questions about the data.

**Instructions**
1. Run the **View dataset statistics** code cell.
1. Inspect the output and answer these questions:
  * What is the maximum fare?
  * What is the mean distance across all trips?
  * How many cab companies are in the dataset?
  * What is the most frequent payment type?
  * Are any features missing data?
1. Run the code **View answers to dataset statistics** code cell to check your answers.


You might be wondering why there are groups of `NaN` (not a number) values listed in the output. When working with data in Python, you may see this value if the result of a calculation can not be computed or if there is missing information. For example, in the taxi dataset `PAYMENT_TYPE` and `COMPANY` are non-numeric, categorical features; numeric information such as mean and max do not make sense for categorical features so the output displays `NaN`.

In [ ]:
#@title Code - View dataset statistics

print('Total number of rows: {0}\n\n'.format(len(training_df.index)))
training_df.describe(include='all')

In [ ]:
#@title Double-click or run to view answers about dataset statistics

answer = '''
What is the maximum fare? 				              Answer: $159.25
What is the mean distance across all trips? 		Answer: 8.2895 miles
How many cab companies are in the dataset? 		  Answer: 31
What is the most frequent payment type? 		    Answer: Credit Card
Are any features missing data? 				          Answer: No
'''

# You should be able to find the answers to the questions about the dataset
# by inspecting the table output after running the DataFrame describe method.
#
# Run this code cell to verify your answers.

# What is the maximum fare?
max_fare = training_df['FARE'].max()
print("What is the maximum fare? 				Answer: ${fare:.2f}".format(fare = max_fare))

# What is the mean distance across all trips?
mean_distance = training_df['TRIP_MILES'].mean()
print("What is the mean distance across all trips? 		Answer: {mean:.4f} miles".format(mean = mean_distance))

# How many cab companies are in the dataset?
num_unique_companies =  training_df['COMPANY'].nunique()
print("How many cab companies are in the dataset? 		Answer: {number}".format(number = num_unique_companies))

# What is the most frequent payment type?
most_freq_payment_type = training_df['PAYMENT_TYPE'].value_counts().idxmax()
print("What is the most frequent payment type? 		Answer: {type}".format(type = most_freq_payment_type))

# Are any features missing data?
missing_values = training_df.isnull().sum().sum()
print("Are any features missing data? 				Answer:", "No" if missing_values == 0 else "Yes")

## Generate a correlation matrix

An important part of machine learning is determining which [features](https://developers.google.com/machine-learning/glossary/#feature) correlate with the [label](https://developers.google.com/machine-learning/glossary/#label). If you have ever taken a taxi ride before, your experience is probably telling you that the fare is typically associated with the distance traveled and the duration of the trip. But, is there a way for you to learn more about how well these features correlate to the fare (label)?

In this step, you will use a **correlation matrix** to identify features whose values correlate well with the label. Correlation values have the following meanings:

  * **`1.0`**: perfect positive correlation; that is, when one attribute rises, the other attribute rises.
  * **`-1.0`**: perfect negative correlation; that is, when one attribute rises, the other attribute falls.
  * **`0.0`**: no correlation; the two columns [are not linearly related](https://en.wikipedia.org/wiki/Correlation_and_dependence#/media/File:Correlation_examples2.svg).

In general, the higher the absolute value of a correlation value, the greater its predictive power.

**Instructions**

1. Inspect the code in the **View correlation matrix** code cell.
1. Run the **View correlation matrix** code cell and inspect the output.
1. **Check your understanding** by answering these questions:
  * Which feature correlates most strongly to the label FARE?
  * Which feature correlates least strongly to the label FARE?


In [ ]:
#@title Code - View correlation matrix
training_df.corr(numeric_only = True)

In [ ]:
#@title Double-click to view answers about the correlation matrix

# Which feature correlates most strongly to the label FARE?
# ---------------------------------------------------------
answer = '''
The feature with the strongest correlation to the FARE is TRIP_MILES.
As you might expect, TRIP_MILES looks like a good feature to start with to train
the model. Also, notice that the feature TRIP_SECONDS has a strong correlation
with fare too.
'''
print(answer)


# Which feature correlates least strongly to the label FARE?
# -----------------------------------------------------------
answer = '''The feature with the weakest correlation to the FARE is TIP_RATE.'''
print(answer)

## Visualize relationships in dataset

Sometimes it is helpful to visualize relationships between features in a dataset; one way to do this is with a pair plot. A **pair plot** generates a grid of pairwise plots to visualize the relationship of each feature with all other features all in one place.

**Instructions**
1. Run the **View pair plot** code cell.

In [ ]:
#@title Code - View pairplot
px.scatter_matrix(training_df, dimensions=["FARE", "TRIP_MILES", "TRIP_SECONDS"])

# Part 3 - Train Model


---

## Define functions to build and train a model

The code you need to build and train your model is in the **Define ML functions** code cell. If you would like to explore this code, expand the code cell and take a look.

**Instructions**
1. Run the **Define ML functions** code cell.

In [ ]:
#@title Code - Define ML functions

def create_model(
    settings: ml_edu.experiment.ExperimentSettings,
    metrics: list[keras.metrics.Metric],
) -> keras.Model:
  """Create and compile a simple linear regression model."""
  # Describe the topography of the model.
  # The topography of a simple linear regression model
  # is a single node in a single layer.
  inputs = {name: keras.Input(shape=(1,), name=name) for name in settings.input_features}
  concatenated_inputs = keras.layers.Concatenate()(list(inputs.values()))
  outputs = keras.layers.Dense(units=1)(concatenated_inputs)
  model = keras.Model(inputs=inputs, outputs=outputs)

  # Compile the model topography into code that Keras can efficiently
  # execute. Configure training to minimize the model's mean squared error.
  model.compile(optimizer=keras.optimizers.RMSprop(learning_rate=settings.learning_rate),
                loss="mean_squared_error",
                metrics=metrics)

  return model


def train_model(
    experiment_name: str,
    model: keras.Model,
    dataset: pd.DataFrame,
    label_name: str,
    settings: ml_edu.experiment.ExperimentSettings,
) -> ml_edu.experiment.Experiment:
  """Train the model by feeding it data."""

  # Feed the model the feature and the label.
  # The model will train for the specified number of epochs.
  features = {name: dataset[name].values for name in settings.input_features}
  label = dataset[label_name].values
  history = model.fit(x=features,
                      y=label,
                      batch_size=settings.batch_size,
                      epochs=settings.number_epochs)

  return ml_edu.experiment.Experiment(
      name=experiment_name,
      settings=settings,
      model=model,
      epochs=history.epoch,
      metrics_history=pd.DataFrame(history.history),
  )

print("SUCCESS: defining linear regression functions complete.")

## Train a model with one feature

In this step you will train a model to predict the cost of the fare using a **single feature**. Earlier, you saw that `TRIP_MILES` (distance) correlates most strongly with the ``FARE``, so let's start with `TRIP_MILES` as the feature for your first training run.

**Instructions**

1. Run the **Experiment 1** code cell to build your model with one feature.
1. Review the output from the training run
1. **Check your understanding** by answering these questions:
  * How many epochs did it take to converge on the final model?
  * How well does the model fit the sample data?

During training, you should see the root mean square error (RMSE) in the output. The units for RMSE are the same as the units for the label (dollars). In other words, you can use the RMSE to determine how far off, on average, the predicted fares are in dollars from the observed values.

In [ ]:
#@title Code - Experiment 1

# The following variables are the hyperparameters.
settings_1 = ml_edu.experiment.ExperimentSettings(
    learning_rate = 0.001,
    number_epochs = 20,
    batch_size = 50,
    input_features = ['TRIP_MILES']
)

metrics = [keras.metrics.RootMeanSquaredError(name='rmse')]

model_1 = create_model(settings_1, metrics)

experiment_1 = train_model('one_feature', model_1, training_df, 'FARE', settings_1)

ml_edu.results.plot_experiment_metrics(experiment_1, ['rmse'])
ml_edu.results.plot_model_predictions(experiment_1, training_df, 'FARE')

In [ ]:
#@title Double-click to view answers for training model with one feature

# How many epochs did it take to converge on the final model?
# -----------------------------------------------------------------------------
answer = """
Use the loss curve to see where the loss begins to level off during training.

With this set of hyperparameters:

  learning_rate = 0.001
  epochs = 20
  batch_size = 50

it takes about 5 epochs for the training run to converge to the final model.
"""
print(answer)

# How well does the model fit the sample data?
# -----------------------------------------------------------------------------
answer = '''
It appears from the model plot that the model fits the sample data fairly well.
'''
print(answer)

## Experiment with hyperparameters

It is common with machine learning to run multiple experiments to find the best set of hyperparmeters to train your model. In this step, try varying the hyperparameters one by one with this set of experiments:

* *Experiment 1:* **Increase** the learning rate to **``1.0``** (batch size at ``50``).
* *Experiment 2:* **Decrease** the learning rate to **``0.0001``** (batch size at ``50``).
* *Experiment 3:* **Increase** the batch size to **``500``** (learning rate at ``0.001``).

**Instructions**
1. Update the hyperparameter values in the **Experiment 2** code cell according to the experiment.
2. Run the **Experiment 2** code cell.
3. After the training run, examine the output and note any differences you see in the loss curve or model output.
4. Repeat steps 1 - 3 for each hyperparameter experiment.
5. **Check your understanding** by answering these questions:
  * How did raising the learning rate impact your ability to train the model?
  * How did lowering the learning rate impact your ability to train the model?
  * Did changing the batch size effect your training results?


**Completed TODO.** The instructions ask for three experiments, changing one hyperparameter at a time.
Instead of editing one cell three times, each experiment has its own cell so all three results stay visible.

In [ ]:
#@title Code - Experiment 2a: learning rate raised to 1.0

settings_2a = ml_edu.experiment.ExperimentSettings(
    learning_rate = 1.0,
    number_epochs = 20,
    batch_size = 50,
    input_features = ['TRIP_MILES']
)

metrics = [keras.metrics.RootMeanSquaredError(name='rmse')]

model_2a = create_model(settings_2a, metrics)

experiment_2a = train_model('lr_1.0', model_2a, training_df, 'FARE', settings_2a)

ml_edu.results.plot_experiment_metrics(experiment_2a, ['rmse'])
ml_edu.results.plot_model_predictions(experiment_2a, training_df, 'FARE')

In [ ]:
#@title Code - Experiment 2b: learning rate lowered to 0.0001

settings_2b = ml_edu.experiment.ExperimentSettings(
    learning_rate = 0.0001,
    number_epochs = 20,
    batch_size = 50,
    input_features = ['TRIP_MILES']
)

metrics = [keras.metrics.RootMeanSquaredError(name='rmse')]

model_2b = create_model(settings_2b, metrics)

experiment_2b = train_model('lr_0.0001', model_2b, training_df, 'FARE', settings_2b)

ml_edu.results.plot_experiment_metrics(experiment_2b, ['rmse'])
ml_edu.results.plot_model_predictions(experiment_2b, training_df, 'FARE')

In [ ]:
#@title Code - Experiment 2c: batch size raised to 500

settings_2c = ml_edu.experiment.ExperimentSettings(
    learning_rate = 0.001,
    number_epochs = 20,
    batch_size = 500,
    input_features = ['TRIP_MILES']
)

metrics = [keras.metrics.RootMeanSquaredError(name='rmse')]

model_2c = create_model(settings_2c, metrics)

experiment_2c = train_model('batch_500', model_2c, training_df, 'FARE', settings_2c)

ml_edu.results.plot_experiment_metrics(experiment_2c, ['rmse'])
ml_edu.results.plot_model_predictions(experiment_2c, training_df, 'FARE')

In [ ]:
#@title Code - Compare the hyperparameter experiments

# Final-epoch training RMSE for the baseline (Experiment 1) and the three variations.
hyper_summary = pd.DataFrame(
    [
        {
            'experiment': exp.name,
            'learning_rate': exp.settings.learning_rate,
            'batch_size': exp.settings.batch_size,
            'first_epoch_rmse': exp.metrics_history['rmse'].iloc[0],
            'final_epoch_rmse': exp.get_final_metric_value('rmse'),
        }
        for exp in [experiment_1, experiment_2a, experiment_2b, experiment_2c]
    ]
).round(4)
hyper_summary

In [ ]:
#@title Double-click to view answers for hyperparameter experiments

# How did raising the learning rate impact your ability to train the model?
# -----------------------------------------------------------------------------
answer = """
When the learning rate is too high, the loss curve bounces around and does not
appear to be moving towards convergence with each iteration. Also, notice that
the predicted model does not fit the data very well. With a learning rate that
is too high, it is unlikely that you will be able to train a model with good
results.
"""
print(answer)

# How did lowering the learning rate impact your ability to train the model?
# -----------------------------------------------------------------------------
answer = '''
When the learning rate is too small, it may take longer for the loss curve to
converge. With a small learning rate the loss curve decreases slowly, but does
not show a dramatic drop or leveling off. With a small learning rate you could
increase the number of epochs so that your model will eventually converge, but
it will take longer.
'''
print(answer)

# Did changing the batch size effect your training results?
# -----------------------------------------------------------------------------
answer = '''
Increasing the batch size makes each epoch run faster, but as with the smaller
learning rate, the model does not converge with just 20 epochs. If you have
time, try increasing the number of epochs and eventually you should see the
model converge.
'''
print(answer)

## Train a model with two features

The model you trained with the feature ``TOTAL_MILES`` demonstrates fairly strong predictive power, but is it possible to do better? In this step, try training the model with two features, ``TRIP_MILES`` and ``TRIP_MINUTES``, to see if you can improve the model. You may recall that the original dataset does not include a feature ``TRIP_MINUTES``, but this feature can be easily derived from ``TRIP_SECONDS`` as shown in the code below.*

**Instructions**
1. Review the code in **Experiment 3** code cell.
1. Run the **Experiment 3** code cell.
1. Review the output from the training run and answer these questions:
  * Does the model with two features produce better results than one using a single feature?
  * Does it make a difference if you use ``TRIP_SECONDS`` instead of ``TRIP_MINUTES``?
  * How well do you think the model comes to the ground truth fare calculation for Chicago Taxi Trips?


Notice that the scatter plot of the features vs. the label is a three dimensional (3-D) plot. This representation allows you to visualize both features and the label all together. The two features (TRIP_MILES and TRIP_MINUTES) are on the x and y axis, and the label (FARE) is on the z axis. The plot shows individual examples in the dataset as circles, and the model as a surface (plane). With this 3-D model, if the trained model is good you would expect most of the examples to land on the plane surface. The 3-D plot is interactive so you can explore the data further by clicking or dragging the plot.

In [ ]:
#@title Code - Experiment 3

# The following variables are the hyperparameters.
settings_3 = ml_edu.experiment.ExperimentSettings(
    learning_rate = 0.001,
    number_epochs = 20,
    batch_size = 50,
    input_features = ['TRIP_MILES', 'TRIP_MINUTES']
)

training_df['TRIP_MINUTES'] = training_df['TRIP_SECONDS']/60

metrics = [keras.metrics.RootMeanSquaredError(name='rmse')]

model_3 = create_model(settings_3, metrics)

experiment_3 = train_model('two_features', model_3, training_df, 'FARE', settings_3)

ml_edu.results.plot_experiment_metrics(experiment_3, ['rmse'])
ml_edu.results.plot_model_predictions(experiment_3, training_df, 'FARE')

In [ ]:
#@title Double-click to view answers for training with two features

# Does the model with two features produce better results than one using a
# single feature?
# -----------------------------------------------------------------------------
answer = '''
To answer this question for your specific training runs, compare the RMSE for
each model. For example, if the RMSE for the model trained with one feature was
3.7457 and the RMSE for the model with two features is 3.4787, that means that
on average the model with two features makes predictions that are about $0.27
closer to the observed fare.

'''
print(answer)

# Does it make a difference if you use TRIP_SECONDS instead of TRIP_MINUTES?
# -----------------------------------------------------------------------------
answer = '''
When training a model with more than one feature, it is important that all
numeric values are roughly on the same scale. In this case, TRIP_SECONDS and
TRIP_MILES do not meet this criteria. The mean value for TRIP_MILES is 8.3 and
the mean for TRIP_SECONDS is 1,320; that is two orders of magnitude difference.
In contrast, the mean for TRIP_MINUTES is 22, which is more similar to the scale
of TRIP_MILES (8.3) than TRIP_SECONDS (1,320). Of course, this is not the
only way to scale values before training, but you will learn about that in
another module.
'''
print(answer)

# How well do you think the model comes to the ground truth fare calculation for
# Chicago taxi trips?
# -----------------------------------------------------------------------------
answer = '''
In reality, Chicago taxi cabs use a documented formula to determine cab fares.
For a single passenger paying cash, the fare is calculated like this:

FARE = 2.25 * TRIP_MILES + 0.12 * TRIP_MINUTES + 3.25

Typically with machine learning problems you would not know the 'correct'
formula, but in this case you can use this knowledge to evaluate your model.
Take a look at your model output (the weights and bias) and determine how
well it matches the ground truth fare calculation. You should find that the
model is roughly close to this formula.
'''
print(answer)

## Compare Experiments

In [ ]:
ml_edu.results.compare_experiment([experiment_1, experiment_3], ['rmse'], training_df, training_df['FARE'].values)

# Part 4 - Validate Model


---

## Use the model to make predictions

Now that you have a trained model, you can use the model to make predictions. In practice, you should make predictions on examples that are not used during training. However, for this exercise, you'll just work with a subset of the same training dataset. In another Colab exercise you will explore ways to make predictions on examples not used in training.

**Instructions**

1. Run the **Define functions to make predictions** code cell.
1. Run the **Make predictions** code cell.
1. Review the predictions in the output.
1. **Check your understanding** by answering these questions:
  * How close is the predicted value to the label value? In other words, does your model accurately predict the fare for a taxi ride?

In [ ]:
#@title Code - Define functions to make predictions
def format_currency(x):
  return "${:.2f}".format(x)

def build_batch(df, batch_size):
  batch = df.sample(n=batch_size).copy()
  batch.set_index(np.arange(batch_size), inplace=True)
  return batch

def predict_fare(model, df, features, label, batch_size=50):
  batch = build_batch(df, batch_size)
  predicted_values = model.predict_on_batch(x={name: batch[name].values for name in features})

  data = {"PREDICTED_FARE": [], "OBSERVED_FARE": [], "L1_LOSS": [],
          features[0]: [], features[1]: []}
  for i in range(batch_size):
    predicted = predicted_values[i][0]
    observed = batch.at[i, label]
    data["PREDICTED_FARE"].append(format_currency(predicted))
    data["OBSERVED_FARE"].append(format_currency(observed))
    data["L1_LOSS"].append(format_currency(abs(observed - predicted)))
    data[features[0]].append(batch.at[i, features[0]])
    data[features[1]].append("{:.2f}".format(batch.at[i, features[1]]))

  output_df = pd.DataFrame(data)
  return output_df

def show_predictions(output):
  header = "-" * 80
  banner = header + "\n" + "|" + "PREDICTIONS".center(78) + "|" + "\n" + header
  print(banner)
  print(output)
  return

In [ ]:
#@title Code - Make predictions

output = predict_fare(experiment_3.model, training_df, experiment_3.settings.input_features, 'FARE')
show_predictions(output)

In [ ]:
#@title Double-click to view answers for validate model

# How close is the predicted value to the label value?
# -----------------------------------------------------------------------------
answer = '''
Based on a random sampling of examples, the model seems to do pretty well
predicting the fare for a taxi ride. Most of the predicted values do not vary
significantly from the observed value. You should be able to see this by looking
at the column L1_LOSS = |observed - predicted|.
'''
print(answer)

# Part 5 - Activity 3 extension


---

Part 4 above checks predictions on rows the model was trained on, and the notebook itself says that in
practice you should predict on examples that were **not** used in training. This part does that, and
produces the numbers and plots the Activity 3 questions ask about.

All the plots in this part use matplotlib so they show up when the notebook is viewed on GitHub.

## 5.1 Split the data into training and test sets

80% of the rows are used to train the model and the other 20% are held back. The model never sees the
test rows while it is learning, so its error on them is an honest estimate of how it does on new trips.

`random_state=42` makes the split the same every time the notebook runs.

In [ ]:
#@title Code - Train/test split
import matplotlib.pyplot as plt

SEED = 42
LABEL = 'FARE'

# training_df already has TRIP_MINUTES (added in Experiment 3).
model_df = training_df.copy()

train_df = model_df.sample(frac=0.8, random_state=SEED)
test_df = model_df.drop(train_df.index)

print('Rows in full dataset: {0}'.format(len(model_df)))
print('Rows in training set: {0} ({1:.0%})'.format(len(train_df), len(train_df) / len(model_df)))
print('Rows in test set:     {0} ({1:.0%})'.format(len(test_df), len(test_df) / len(model_df)))
print('Rows in both sets:    {0}'.format(len(train_df.index.intersection(test_df.index))))

## 5.2 Train on the training set, evaluate on the test set

Same model and hyperparameters as Experiment 3 (two features, learning rate 0.001, 20 epochs, batch size 50),
but trained only on the training rows. Besides RMSE the model now also tracks **mean absolute error (MAE)**.

Two helper functions are defined here and reused for the rest of Part 5:

* `run_experiment` – builds and trains a model on `train_df` (it reuses `create_model` from Part 3 and
  trains quietly instead of printing every epoch).
* `score` – returns MSE, RMSE and MAE for a trained model on any dataframe.

In [ ]:
#@title Code - Helper functions for Part 5

pd.set_option('display.width', 200)   # keep printed tables on one line

def run_experiment(name, settings, seed=SEED):
  """Train a model on the training set only and return it as an Experiment."""
  keras.utils.set_random_seed(seed)   # same starting weights and shuffling every run
  metrics = [
      keras.metrics.RootMeanSquaredError(name='rmse'),
      keras.metrics.MeanAbsoluteError(name='mae'),
  ]
  model = create_model(settings, metrics)
  features = {name: train_df[name].values for name in settings.input_features}
  history = model.fit(x=features,
                      y=train_df[LABEL].values,
                      batch_size=settings.batch_size,
                      epochs=settings.number_epochs,
                      verbose=0)
  return ml_edu.experiment.Experiment(
      name=name,
      settings=settings,
      model=model,
      epochs=history.epoch,
      metrics_history=pd.DataFrame(history.history),
  )


def predict(experiment, df):
  """Return the model's predicted fare for every row of df."""
  features = {name: df[name].values for name in experiment.settings.input_features}
  return experiment.model.predict(features, verbose=0).flatten().astype(float)


def score(experiment, df):
  """MSE, RMSE and MAE of the model's predictions on df."""
  error = predict(experiment, df) - df[LABEL].values
  mse = float(np.mean(error ** 2))
  return {'MSE': mse, 'RMSE': mse ** 0.5, 'MAE': float(np.mean(np.abs(error)))}


def describe_model(experiment):
  """Print the learned weights and bias as an equation."""
  weights, bias = experiment.model.get_weights()
  terms = ['{0:.4f} * {1}'.format(w, name)
           for w, name in zip(weights.flatten(), experiment.settings.input_features)]
  print('{0} = {1} + {2:.4f}'.format(LABEL, ' + '.join(terms), bias[0]))

print('SUCCESS: defining Part 5 helper functions complete.')

In [ ]:
#@title Code - Train the final model and score it

settings_final = ml_edu.experiment.ExperimentSettings(
    learning_rate = 0.001,
    number_epochs = 20,
    batch_size = 50,
    input_features = ['TRIP_MILES', 'TRIP_MINUTES']
)

experiment_final = run_experiment('final_model', settings_final)

train_scores = score(experiment_final, train_df)
test_scores = score(experiment_final, test_df)

# Baseline: ignore the features and always guess the average training fare.
baseline_guess = train_df[LABEL].mean()
baseline_mae = float((test_df[LABEL] - baseline_guess).abs().mean())

print('Learned model:')
describe_model(experiment_final)
print('Chicago fare formula from Part 3:')
print('FARE = 2.25 * TRIP_MILES + 0.12 * TRIP_MINUTES + 3.25\n')

print(pd.DataFrame({'training set': train_scores, 'test set': test_scores}).round(4), '\n')
print('Average fare in the test set:          ${0:.2f}'.format(test_df[LABEL].mean()))
print('Test MAE of the model:                 ${0:.2f}'.format(test_scores['MAE']))
print('Test MAE of always guessing ${0:.2f}:    ${1:.2f}'.format(baseline_guess, baseline_mae))

## 5.3 Predicted vs. actual fares

Each dot is one trip from the **test set**. The horizontal position is the fare that was actually charged and
the vertical position is the fare the model predicted. A perfect model would put every dot on the dashed line.

The table under the plot breaks the test set into fare ranges, to show where the error is small and where it
is large. `mean_error` is predicted minus actual, so a negative value means the model guesses too low.

In [ ]:
#@title Code - Predicted vs. actual plot

results_df = test_df[['TRIP_MILES', 'TRIP_MINUTES', LABEL]].copy()
results_df['PREDICTED_FARE'] = predict(experiment_final, test_df)
results_df['ERROR'] = results_df['PREDICTED_FARE'] - results_df[LABEL]
results_df['ABS_ERROR'] = results_df['ERROR'].abs()

axis_max = max(results_df[LABEL].max(), results_df['PREDICTED_FARE'].max()) * 1.05

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(results_df[LABEL], results_df['PREDICTED_FARE'], s=12, alpha=0.25,
           color='#2a78d6', edgecolors='none')
ax.plot([0, axis_max], [0, axis_max], linestyle='--', linewidth=1.5, color='#52514e')
ax.annotate('perfect prediction', xy=(axis_max * 0.80, axis_max * 0.80),
            xytext=(axis_max * 0.52, axis_max * 0.93), color='#52514e',
            arrowprops={'arrowstyle': '-', 'color': '#52514e', 'linewidth': 0.8})
ax.set_xlim(0, axis_max)
ax.set_ylim(0, axis_max)
ax.set_aspect('equal')
ax.set_xlabel('Actual fare ($)')
ax.set_ylabel('Predicted fare ($)')
ax.set_title('Predicted vs. actual fare, test set ({0} trips)'.format(len(results_df)))
ax.grid(alpha=0.3)
ax.set_axisbelow(True)
plt.show()

In [ ]:
#@title Code - Error by fare range, and the worst predictions

fare_ranges = pd.cut(results_df[LABEL],
                     bins=[0, 10, 20, 40, 60, np.inf],
                     labels=['$0-10', '$10-20', '$20-40', '$40-60', 'over $60'],
                     include_lowest=True)

error_by_range = results_df.groupby(fare_ranges, observed=True).agg(
    trips=('ERROR', 'size'),
    mean_abs_error=('ABS_ERROR', 'mean'),
    mean_error=('ERROR', 'mean'),
).round(2)
error_by_range['share_of_trips'] = (error_by_range['trips'] / len(results_df)).round(3)
print('Error by actual fare range (test set)')
print(error_by_range, '\n')

print('The 10 trips the model got most wrong (test set)')
print(results_df.sort_values('ABS_ERROR', ascending=False).head(10).round(2))

## 5.4 Change one setting

Only the **learning rate** is changed, from `0.001` to `0.0001`. The features, number of epochs, batch size,
training rows and random seed all stay the same, so any difference in the results comes from the learning rate.

In [ ]:
#@title Code - Lower the learning rate and compare

settings_low_lr = ml_edu.experiment.ExperimentSettings(
    learning_rate = 0.0001,      # the only setting that changed (was 0.001)
    number_epochs = 20,
    batch_size = 50,
    input_features = ['TRIP_MILES', 'TRIP_MINUTES']
)

experiment_low_lr = run_experiment('low_learning_rate', settings_low_lr)
low_lr_scores = score(experiment_low_lr, test_df)

setting_comparison = pd.DataFrame(
    {'learning_rate = 0.001': test_scores, 'learning_rate = 0.0001': low_lr_scores}
).round(4)
print('Test set results')
print(setting_comparison, '\n')

print('Model learned with learning_rate = 0.001:')
describe_model(experiment_final)
print('Model learned with learning_rate = 0.0001:')
describe_model(experiment_low_lr)

# Training loss curves for the two runs on one axis.
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(np.array(experiment_final.epochs) + 1, experiment_final.metrics_history['rmse'],
        linewidth=2, marker='o', markersize=4, color='#2a78d6', label='learning rate 0.001')
ax.plot(np.array(experiment_low_lr.epochs) + 1, experiment_low_lr.metrics_history['rmse'],
        linewidth=2, marker='s', markersize=4, color='#eb6834', label='learning rate 0.0001')
ax.set_xticks(range(1, settings_final.number_epochs + 1, 2))
ax.set_ylim(bottom=0)
ax.set_xlabel('Epoch')
ax.set_ylabel('Training RMSE ($)')
ax.set_title('Training loss curve for each learning rate')
ax.grid(alpha=0.3)
ax.set_axisbelow(True)
ax.legend(frameon=False)
plt.show()

## 5.5 Which feature matters most?

Three separate checks, so the answer does not depend on one method:

1. **Correlation** of each feature with the fare (training set).
2. **Ablation** – train the model with only one feature at a time and compare the test MAE with the
   two-feature model. The feature that does better on its own, and hurts more when it is removed, matters more.
3. **Weight x spread** – a weight on its own is not a fair comparison because miles and minutes are in
   different units. Multiplying each weight by that feature's standard deviation gives how many dollars the
   prediction moves for a typical change in the feature.

In [ ]:
#@title Code - Feature importance

feature_names = ['TRIP_MILES', 'TRIP_MINUTES']

# 1. Correlation with the label.
correlations = train_df[feature_names + [LABEL]].corr()[LABEL].drop(LABEL)

# 2. Ablation: one model per single feature.
single_feature_experiments = {}
for feature in feature_names:
  settings_single = ml_edu.experiment.ExperimentSettings(
      learning_rate = 0.001,
      number_epochs = 20,
      batch_size = 50,
      input_features = [feature]
  )
  single_feature_experiments[feature] = run_experiment('only_' + feature, settings_single)

ablation = pd.DataFrame({
    'both features': test_scores,
    'TRIP_MILES only': score(single_feature_experiments['TRIP_MILES'], test_df),
    'TRIP_MINUTES only': score(single_feature_experiments['TRIP_MINUTES'], test_df),
    'no features (average fare)': {'MSE': float(((test_df[LABEL] - baseline_guess) ** 2).mean()),
                                   'RMSE': float(((test_df[LABEL] - baseline_guess) ** 2).mean() ** 0.5),
                                   'MAE': baseline_mae},
}).T.round(4)

# 3. Weight x standard deviation for the two-feature model.
final_weights = experiment_final.model.get_weights()[0].flatten()
importance = pd.DataFrame({
    'correlation_with_fare': correlations,
    'weight_dollars_per_unit': pd.Series(final_weights, index=feature_names),
    'feature_std': train_df[feature_names].std(),
})
importance['dollars_per_std'] = importance['weight_dollars_per_unit'] * importance['feature_std']

print('Correlation and weights (two-feature model)')
print(importance.round(4), '\n')
print('Test set error by which features the model is given')
print(ablation)

fig, ax = plt.subplots(figsize=(8, 3.5))
bars = ax.barh(ablation.index[::-1], ablation['MAE'][::-1], height=0.55, color='#2a78d6')
ax.bar_label(bars, labels=['${0:.2f}'.format(v) for v in ablation['MAE'][::-1]], padding=4)
ax.set_xlim(0, ablation['MAE'].max() * 1.15)
ax.set_xlabel('Test MAE ($) - lower is better')
ax.set_title('Test error by which features the model is given')
ax.grid(axis='x', alpha=0.3)
ax.set_axisbelow(True)
for side in ['top', 'right']:
  ax.spines[side].set_visible(False)
plt.show()

## 5.6 Summary of results

One place with every number quoted in `answers.md`.

In [ ]:
#@title Code - Summary

print('Test set size:                         {0} trips'.format(len(test_df)))
print('Average test fare:                     ${0:.2f}'.format(test_df[LABEL].mean()))
print('Final model (lr 0.001)  test MAE:      ${0:.2f}'.format(test_scores['MAE']))
print('Final model (lr 0.001)  test RMSE:     ${0:.2f}'.format(test_scores['RMSE']))
print('Final model (lr 0.001)  test MSE:      {0:.2f}'.format(test_scores['MSE']))
print('Final model (lr 0.001)  training MAE:  ${0:.2f}'.format(train_scores['MAE']))
print('Always guess the average: test MAE:    ${0:.2f}'.format(baseline_mae))
print('Lower learning rate (0.0001) test MAE: ${0:.2f}'.format(low_lr_scores['MAE']))
print('TRIP_MILES only         test MAE:      ${0:.2f}'.format(ablation.loc['TRIP_MILES only', 'MAE']))
print('TRIP_MINUTES only       test MAE:      ${0:.2f}'.format(ablation.loc['TRIP_MINUTES only', 'MAE']))